# Utilisation du modèle avec de nouvelles images

In [1]:
fichier_json = open('modele/modele.json', 'r')
modele_json = fichier_json.read()
fichier_json.close()

from keras.models import model_from_json
modele = model_from_json(modele_json)

modele.load_weights("modele/modele.weights.h5")

I0000 00:00:1794294047.692984    3032 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794294047.707427    3032 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794294050.195964    3032 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794294058.618817    3032 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
classes = ["Un T-shirt/haut", "Un pantalon", "Un pull", "Une robe", "Un manteau", "Des sandales", "Une chemise", "Des basckets", "Un sac", "Des bottes de cheville"]

In [5]:
import numpy as np
from PIL import Image, ImageFilter

def load_and_preprocess_image(image_path, target_size=28, content_size=20):
    image = Image.open(image_path).convert('L')
    resized_img = resize_with_aspect_ratio(image, content_size)
    canvas = create_centered_image(resized_img, target_size)
    normalized_array = normalize_pixels(canvas)

    return normalized_array.reshape(1, target_size, target_size, 1)

def resize_with_aspect_ratio(image, target_dimension):
    width, height = image.size

    if width > height:
        new_width = target_dimension
        new_height = max(1, round(target_dimension * height / width))
    else:
        new_height = target_dimension
        new_width = max(1, round(target_dimension * width / height))

    return image.resize((new_width, new_height), Image.LANCZOS).filter(ImageFilter.SHARPEN)

def create_centered_image(image, canvas_size):
    canvas = Image.new('L', (canvas_size, canvas_size), 255)

    img_width, img_height = image.size

    x_offset = (canvas_size - img_width) // 2
    y_offset = (canvas_size - img_height) // 2

    canvas.paste(image, (x_offset, y_offset))
    return canvas

def normalize_pixels(image):
    pixels = np.array(image.getdata(), dtype=np.float32)
    return (255 - pixels) / 255.0

In [8]:
image_array = load_and_preprocess_image("image.png")

probabilities = modele.predict(image_array)[0]
predicted_class = np.argmax(probabilities)

print(f"Selon moi l'image est : {classes[predicted_class]}\n")

for i, class_name in enumerate(classes):
    print(f"{class_name}: {probabilities[i] * 100:.2f}%")

/tmp/ipykernel_3032/145950218.py:36: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = np.array(image.getdata(), dtype=np.float32)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Selon moi l'image est : Un pantalon

Un T-shirt/haut: 1.01%
Un pantalon: 93.76%
Un pull: 0.07%
Une robe: 2.85%
Un manteau: 0.00%
Des sandales: 0.12%
Une chemise: 0.03%
Des basckets: 0.00%
Un sac: 2.15%
Des bottes de cheville: 0.00%
